In [ ]:
!pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("first_25000_rows.csv")
df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df.sort_values('ts_event').reset_index(drop=True)
df.head()

In [ ]:
def compute_best_level_ofi(df):
    best_ofi = []
    for i in range(1, len(df)):
        bid_same = df.loc[i, 'bid_px_00'] == df.loc[i-1, 'bid_px_00']
        ask_same = df.loc[i, 'ask_px_00'] == df.loc[i-1, 'ask_px_00']

        delta_bid = df.loc[i, 'bid_sz_00'] - df.loc[i-1, 'bid_sz_00'] if bid_same else 0
        delta_ask = df.loc[i, 'ask_sz_00'] - df.loc[i-1, 'ask_sz_00'] if ask_same else 0

        best_ofi.append(delta_bid - delta_ask)
    return pd.Series(best_ofi, name='OFI_best_level')

def compute_multi_level_ofi(df, levels=5):
    multi_ofi = []
    for i in range(1, len(df)):
        ofi = 0
        for lvl in range(levels):
            bid_px_col = f'bid_px_0{lvl}'
            bid_sz_col = f'bid_sz_0{lvl}'
            ask_px_col = f'ask_px_0{lvl}'
            ask_sz_col = f'ask_sz_0{lvl}'

            bid_same = df.loc[i, bid_px_col] == df.loc[i-1, bid_px_col]
            ask_same = df.loc[i, ask_px_col] == df.loc[i-1, ask_px_col]

            delta_bid = df.loc[i, bid_sz_col] - df.loc[i-1, bid_sz_col] if bid_same else 0
            delta_ask = df.loc[i, ask_sz_col] - df.loc[i-1, ask_sz_col] if ask_same else 0

            ofi += delta_bid - delta_ask
        multi_ofi.append(ofi)
    return pd.Series(multi_ofi, name='OFI_multi_level')

def compute_integrated_ofi(df, levels=10):
    pca_data = []
    for i in range(1, len(df)):
        row = []
        for lvl in range(levels):
            bid_px_col = f'bid_px_0{lvl}'
            bid_sz_col = f'bid_sz_0{lvl}'
            ask_px_col = f'ask_px_0{lvl}'
            ask_sz_col = f'ask_sz_0{lvl}'

            bid_same = df.loc[i, bid_px_col] == df.loc[i-1, bid_px_col]
            ask_same = df.loc[i, ask_px_col] == df.loc[i-1, ask_px_col]

            delta_bid = df.loc[i, bid_sz_col] - df.loc[i-1, bid_sz_col] if bid_same else 0
            delta_ask = df.loc[i, ask_sz_col] - df.loc[i-1, ask_sz_col] if ask_same else 0

            row.append(delta_bid - delta_ask)
        pca_data.append(row)

    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(pca_data)
    return pd.Series(pca_result.flatten(), name='OFI_integrated')


In [ ]:
timestamps = df['ts_event'][1:].reset_index(drop=True)

ofi_best = compute_best_level_ofi(df)
ofi_multi = compute_multi_level_ofi(df, levels=5)
ofi_integrated = compute_integrated_ofi(df, levels=10)

# Since only one stock (AAPL), Cross-Asset OFI is NaN
ofi_cross_asset = pd.Series([np.nan] * len(ofi_best), name="OFI_cross_asset")

# Combine everything into one DataFrame
ofi_df = pd.DataFrame({
    "timestamp": timestamps,
    "OFI_best_level": ofi_best,
    "OFI_multi_level": ofi_multi,
    "OFI_integrated": ofi_integrated,
    "OFI_cross_asset": ofi_cross_asset
})
ofi_df.head()
